# Hasil eksekusi Colab: 04_model_library

Dijalankan di VM Colab GPU T4 (sesi `d920cd`). Sel di bawah adalah kode notebook asli beserta outputnya.


In [ ]:
import pandas as pd
import numpy as np
import warnings
warnings.filterwarnings('ignore')

from sklearn.model_selection import train_test_split, cross_val_score
from sklearn.preprocessing import StandardScaler
from sklearn.linear_model import LogisticRegression
from sklearn.tree import DecisionTreeClassifier
from sklearn.ensemble import RandomForestClassifier
from sklearn.svm import SVC
from sklearn.metrics import accuracy_score, precision_score, recall_score, f1_score, confusion_matrix, classification_report

URL = "https://raw.githubusercontent.com/ravi-arnan/machine-learning/main/data/healthcare-stroke-data.csv"
df = pd.read_csv(URL)

df = df[["age", "hypertension", "heart_disease", "avg_glucose_level", "bmi", "stroke"]].copy()
df["bmi"] = df["bmi"].fillna(df["bmi"].median())

print("Ukuran dataset:", df.shape)
print()
print("5 baris pertama:")
df.head()

Ukuran dataset: (5110, 6)

5 baris pertama:


,age,hypertension,heart_disease,avg_glucose_level,bmi,stroke
0,67.0,0,1,228.69,36.6,1
1,61.0,0,0,202.21,28.1,1
2,80.0,0,1,105.92,32.5,1
3,49.0,0,0,171.23,34.4,1
4,79.0,1,0,174.12,24.0,1


In [ ]:
X = df.drop(columns=["stroke"]).values
y = df["stroke"].values

X_train, X_test, y_train, y_test = train_test_split(
    X, y, test_size=0.2, stratify=y, random_state=42
)

print(f"Train: {X_train.shape[0]} baris")
print(f"Test:  {X_test.shape[0]} baris")

# Scaling untuk model yang sensitif skala (Logistic Regression, SVM, KNN)
scaler = StandardScaler()
X_train_scaled = scaler.fit_transform(X_train)
X_test_scaled = scaler.transform(X_test)

print("\nProporsi stroke:")
print(f"  Train: {y_train.mean():.3f}")
print(f"  Test:  {y_test.mean():.3f}")

Train: 4088 baris
Test:  1022 baris

Proporsi stroke:
  Train: 0.049
  Test:  0.049


In [ ]:
def evaluasi_model(model, X_test, y_test):
    """Prediksi dan cetak metrik."""
    y_pred = model.predict(X_test)

    hasil = {
        "akurasi": round(accuracy_score(y_test, y_pred), 4),
        "presisi": round(precision_score(y_test, y_pred, zero_division=0), 4),
        "recall": round(recall_score(y_test, y_pred, zero_division=0), 4),
        "f1": round(f1_score(y_test, y_pred, zero_division=0), 4),
    }
    tn, fp, fn, tp = confusion_matrix(y_test, y_pred).ravel()
    hasil["tp"] = tp
    hasil["tn"] = tn
    hasil["fp"] = fp
    hasil["fn"] = fn
    return hasil, y_pred


def cetak_hasil(nama, hasil):
    print(f"\n{'=' * 50}")
    print(f"  {nama}")
    print(f"{'=' * 50}")
    print(f"  Akurasi:  {hasil['akurasi']}")
    print(f"  Presisi:  {hasil['presisi']}")
    print(f"  Recall:   {hasil['recall']}")
    print(f"  F1-score: {hasil['f1']}")
    print(f"  TP={hasil['tp']}  FP={hasil['fp']}  FN={hasil['fn']}  TN={hasil['tn']}")

In [ ]:
# Tanpa scaling vs dengan scaling
for nama, X_tr, X_te in [
    ("Logistic Regression (tanpa scaling)", X_train, X_test),
    ("Logistic Regression (dengan scaling)", X_train_scaled, X_test_scaled),
]:
    for imbang in [None, "balanced"]:
        label = f"{nama}, class_weight={imbang}"
        model = LogisticRegression(max_iter=5000, class_weight=imbang)
        model.fit(X_tr, y_train)
        hasil, _ = evaluasi_model(model, X_te, y_test)
        cetak_hasil(label, hasil)


  Logistic Regression (tanpa scaling), class_weight=None
  Akurasi:  0.9511
  Presisi:  0.0
  Recall:   0.0
  F1-score: 0.0
  TP=0  FP=0  FN=50  TN=972

  Logistic Regression (tanpa scaling), class_weight=balanced
  Akurasi:  0.7446
  Presisi:  0.1375
  Recall:   0.8
  F1-score: 0.2346
  TP=40  FP=251  FN=10  TN=721

  Logistic Regression (dengan scaling), class_weight=None
  Akurasi:  0.9511
  Presisi:  0.0
  Recall:   0.0
  F1-score: 0.0
  TP=0  FP=0  FN=50  TN=972

  Logistic Regression (dengan scaling), class_weight=balanced
  Akurasi:  0.7446
  Presisi:  0.1375
  Recall:   0.8
  F1-score: 0.2346
  TP=40  FP=251  FN=10  TN=721


In [ ]:
for kedalaman in [3, 5, 10, None]:
    for imbang in [None, "balanced"]:
        label = f"Decision Tree (max_depth={kedalaman}, class_weight={imbang})"
        model = DecisionTreeClassifier(max_depth=kedalaman, class_weight=imbang, random_state=42)
        model.fit(X_train, y_train)
        hasil, _ = evaluasi_model(model, X_test, y_test)
        cetak_hasil(label, hasil)


  Decision Tree (max_depth=3, class_weight=None)
  Akurasi:  0.9491
  Presisi:  0.0
  Recall:   0.0
  F1-score: 0.0
  TP=0  FP=2  FN=50  TN=970

  Decision Tree (max_depth=3, class_weight=balanced)
  Akurasi:  0.6526
  Presisi:  0.108
  Recall:   0.84
  F1-score: 0.1913
  TP=42  FP=347  FN=8  TN=625

  Decision Tree (max_depth=5, class_weight=None)
  Akurasi:  0.9472
  Presisi:  0.0
  Recall:   0.0
  F1-score: 0.0
  TP=0  FP=4  FN=50  TN=968

  Decision Tree (max_depth=5, class_weight=balanced)
  Akurasi:  0.8209
  Presisi:  0.1658
  Recall:   0.66
  F1-score: 0.2651
  TP=33  FP=166  FN=17  TN=806

  Decision Tree (max_depth=10, class_weight=None)
  Akurasi:  0.9266
  Presisi:  0.037
  Recall:   0.02
  F1-score: 0.026
  TP=1  FP=26  FN=49  TN=946

  Decision Tree (max_depth=10, class_weight=balanced)
  Akurasi:  0.8229
  Presisi:  0.1124
  Recall:   0.38
  F1-score: 0.1735
  TP=19  FP=150  FN=31  TN=822

  Decision Tree (max_depth=None, class_weight=None)
  Akurasi:  0.9119
  Presisi:

In [ ]:
for n in [50, 100, 200]:
    for imbang in [None, "balanced"]:
        label = f"Random Forest (n={n}, class_weight={imbang})"
        model = RandomForestClassifier(n_estimators=n, class_weight=imbang, random_state=42)
        model.fit(X_train, y_train)
        hasil, _ = evaluasi_model(model, X_test, y_test)
        cetak_hasil(label, hasil)


  Random Forest (n=50, class_weight=None)
  Akurasi:  0.9501
  Presisi:  0.0
  Recall:   0.0
  F1-score: 0.0
  TP=0  FP=1  FN=50  TN=971



  Random Forest (n=50, class_weight=balanced)
  Akurasi:  0.9521
  Presisi:  1.0
  Recall:   0.02
  F1-score: 0.0392
  TP=1  FP=0  FN=49  TN=972



  Random Forest (n=100, class_weight=None)
  Akurasi:  0.9491
  Presisi:  0.0
  Recall:   0.0
  F1-score: 0.0
  TP=0  FP=2  FN=50  TN=970



  Random Forest (n=100, class_weight=balanced)
  Akurasi:  0.9521
  Presisi:  1.0
  Recall:   0.02
  F1-score: 0.0392
  TP=1  FP=0  FN=49  TN=972



  Random Forest (n=200, class_weight=None)
  Akurasi:  0.9481
  Presisi:  0.0
  Recall:   0.0
  F1-score: 0.0
  TP=0  FP=3  FN=50  TN=969



  Random Forest (n=200, class_weight=balanced)
  Akurasi:  0.9521
  Presisi:  1.0
  Recall:   0.02
  F1-score: 0.0392
  TP=1  FP=0  FN=49  TN=972


In [ ]:
for kernel in ["linear", "rbf"]:
    for imbang in [None, "balanced"]:
        label = f"SVM ({kernel}, class_weight={imbang})"
        model = SVC(kernel=kernel, class_weight=imbang, random_state=42)
        model.fit(X_train_scaled, y_train)
        hasil, _ = evaluasi_model(model, X_test_scaled, y_test)
        cetak_hasil(label, hasil)


  SVM (linear, class_weight=None)
  Akurasi:  0.9511
  Presisi:  0.0
  Recall:   0.0
  F1-score: 0.0
  TP=0  FP=0  FN=50  TN=972



  SVM (linear, class_weight=balanced)
  Akurasi:  0.727
  Presisi:  0.1294
  Recall:   0.8
  F1-score: 0.2228
  TP=40  FP=269  FN=10  TN=703

  SVM (rbf, class_weight=None)
  Akurasi:  0.9511
  Presisi:  0.0
  Recall:   0.0
  F1-score: 0.0
  TP=0  FP=0  FN=50  TN=972



  SVM (rbf, class_weight=balanced)
  Akurasi:  0.6908
  Presisi:  0.1178
  Recall:   0.82
  F1-score: 0.206
  TP=41  FP=307  FN=9  TN=665


In [ ]:
model_terbaik = [
    ("Logistic Regression + scaling + balanced",
     LogisticRegression(max_iter=5000, class_weight="balanced").fit(X_train_scaled, y_train),
     X_test_scaled),
    ("Decision Tree (max_depth=5, balanced)",
     DecisionTreeClassifier(max_depth=5, class_weight="balanced", random_state=42).fit(X_train, y_train),
     X_test),
    ("Random Forest (n=100, balanced)",
     RandomForestClassifier(n_estimators=100, class_weight="balanced", random_state=42).fit(X_train, y_train),
     X_test),
    ("SVM (rbf, balanced)",
     SVC(kernel="rbf", class_weight="balanced", random_state=42).fit(X_train_scaled, y_train),
     X_test_scaled),
]

print(f"{'Model':<45} {'Akurasi':>8} {'Presisi':>8} {'Recall':>8} {'F1':>8}")
print("-" * 77)

for nama, model, X_te in model_terbaik:
    hasil, _ = evaluasi_model(model, X_te, y_test)
    print(f"{nama:<45} {hasil['akurasi']:>8} {hasil['presisi']:>8} {hasil['recall']:>8} {hasil['f1']:>8}")

Model                                          Akurasi  Presisi   Recall       F1
-----------------------------------------------------------------------------
Logistic Regression + scaling + balanced        0.7446   0.1375      0.8   0.2346
Decision Tree (max_depth=5, balanced)           0.8209   0.1658     0.66   0.2651
Random Forest (n=100, balanced)                 0.9521      1.0     0.02   0.0392
SVM (rbf, balanced)                             0.6908   0.1178     0.82    0.206


In [ ]:
print("Kesimpulan:")
print()
print("Karena dataset tidak seimbang (stroke hanya 5%), metrik utama adalah F1-score,")
print("bukan akurasi. Model terbaik adalah yang punya F1 tertinggi.")
print()
print("Catatan: Random Forest dan SVM dengan kernel rbf cenderung lebih baik")
print("untuk dataset dengan hubungan non-linear dan class imbalance.")
print()
print("Cross-validation dan hyperparameter tuning lebih lanjut bisa dilakukan")

Kesimpulan:

Karena dataset tidak seimbang (stroke hanya 5%), metrik utama adalah F1-score,
bukan akurasi. Model terbaik adalah yang punya F1 tertinggi.

Catatan: Random Forest dan SVM dengan kernel rbf cenderung lebih baik
untuk dataset dengan hubungan non-linear dan class imbalance.

Cross-validation dan hyperparameter tuning lebih lanjut bisa dilakukan
